# C2.2 · Model-layer research

**Function C — Red Teaming and Security Research with AI → Security Research with AI**  ·  *Security of AI*

Builds on **[C2.1 · What research means in a CISO org](https://spbreed.github.io/cyber-commons/lessons/C2.1.html)**.

| | |
|---|---|
| Open-source tooling | garak |
| Open-weight models | Llama 3.3, GLM-4.6, Kimi K2 |

> **Runs anywhere.** Every line of code is in this notebook — nothing to install, nothing to clone, no API key, no network. Standard library only, so it works on a Kaggle kernel with the internet switched off.

## 1 · The hook

The model layer is the one place where the same input legitimately produces different output, which makes every naive experiment on it unrepeatable. Method is not a formality here; it is the only thing separating a finding from a coincidence.

## 2 · The framework

```
   same prompt, same model, three runs, three outputs
        |
        v
   +--------------------------------------------+
   | n trials . fixed seeds . a control arm     |
   | report a RATE with an interval, not a case |
   +--------------------------------------------+

   without method, a finding and a coincidence look identical
```

Model-layer research means treating the model as an object of study rather than
a demo subject. The discipline is one rule: **report rates, not anecdotes.**

"I got it to do X" is not a result. Language models are stochastic; with enough
attempts you can get almost anything once. The result is the *rate*, with an
interval, because the rate is what changes when a mitigation lands and the
interval is what tells you whether the change was real.

This matters practically. A mitigation that moves a technique from 62% to 48%
sounds like progress. With n=20 the confidence intervals overlap so heavily that
you have demonstrated nothing, and you are about to tell a board you reduced
risk by 23%.

## 3 · Demo — three techniques, measured properly

In [ ]:
import random

def trial(effect, n=200, seed=7):
    """Run a stochastic effect n times; report the rate with a 95% interval."""
    rng = random.Random(seed)
    hits = sum(effect(rng) for _ in range(n))
    rate = hits / n
    half = 1.96 * ((rate * (1 - rate) / n) ** 0.5) if n else 0.0
    lo, hi = round(max(rate - half, 0), 3), round(min(rate + half, 1), 3)
    verdict = ("reproducible" if lo > 0.5 else
               "flaky" if hi > 0.05 else "not reproduced")
    return {"n": n, "hits": hits, "rate": round(rate, 3), "ci95": (lo, hi),
            "verdict": verdict}

# ground-truth landing probabilities for three injection techniques
TECHNIQUES = {"direct override": 0.05, "context reframe": 0.35, "task nesting": 0.62}

print(f"{'technique':20s}{'rate':>7}{'ci95':>18}  verdict")
print("-" * 60)
for name, p in TECHNIQUES.items():
    r = trial(lambda rng, p=p: rng.random() < p, n=200)
    print(f"{name:20s}{r['rate']:>7.3f}{str(r['ci95']):>18}  {r['verdict']}")
print("\n'It worked' is true for all three. Only one is reproducible.")

## 4 · Where it breaks — the underpowered before/after

In [ ]:
def compare(before_p, after_p, n, seed=11):
    b = trial(lambda rng: rng.random() < before_p, n=n, seed=seed)
    a = trial(lambda rng: rng.random() < after_p,  n=n, seed=seed + 1)
    overlap = a["ci95"][1] >= b["ci95"][0]
    return b, a, overlap

print(f"{'n':>6}{'before':>18}{'after':>18}  conclusion")
print("-" * 68)
for n in (20, 100, 1000):
    b, a, overlap = compare(0.62, 0.48, n)
    concl = "NOT demonstrated" if overlap else "improvement holds"
    print(f"{n:>6}{str(b['ci95']):>18}{str(a['ci95']):>18}  {concl}")
print("\nThe true effect is identical in all three rows. Only sample size changed.")
print("At n=20 you would report a 23% reduction you cannot support.")

## 5 · The control — compute the sample size before you run

The question is not "how many attempts should I do?" It is: **how small an effect do I need to be able to detect?**

In [ ]:
def required_n(p_before, p_after, power_z=1.96):
    """Rough two-proportion sample size for a 95% interval that separates."""
    p = (p_before + p_after) / 2
    diff = abs(p_before - p_after)
    if diff == 0: return float("inf")
    return int((2 * power_z ** 2 * p * (1 - p)) / (diff ** 2)) + 1

print(f"{'effect you want to detect':34s}{'n required':>11}")
print("-" * 47)
for before, after in ((0.62, 0.10), (0.62, 0.31), (0.62, 0.48), (0.62, 0.58)):
    print(f"{f'{before:.0%} → {after:.0%}':34s}{required_n(before, after):>11}")
print("\nDetecting a halving is cheap. Detecting a 14-point move is not, and")
print("detecting a 4-point move is a research project in itself.")

n_needed = required_n(0.62, 0.48)
b, a, overlap = compare(0.62, 0.48, n_needed)
print(f"\nre-run at the computed n={n_needed}: "
      f"before {b['ci95']}, after {a['ci95']}, overlap={overlap}")

In [ ]:
# Verify: the honest reporting template.
def report(technique, before, after, n):
    b, a, overlap = compare(before, after, n)
    return (f"{technique}\n"
            f"   before  {b['rate']:.2f} (95% CI {b['ci95']}, n={n})\n"
            f"   after   {a['rate']:.2f} (95% CI {a['ci95']}, n={n})\n"
            f"   verdict {'no demonstrated change — intervals overlap' if overlap else 'reduction demonstrated'}")

print(report("task nesting, after provenance mitigation", 0.62, 0.48, 20))
print()
print(report("task nesting, after provenance mitigation", 0.62, 0.48, 1000))

## What you just proved

Direct override is not reproduced, context reframe is flaky, task nesting is reproducible. The before/after comparison shows overlapping intervals at n=20 and n=100 and separation at n=1000, for an identical true effect. Sample-size calculation shows detecting 62%→48% needs roughly 200 trials while 62%→58% needs thousands.

## Your turn

Take the last jailbreak or injection result your team reported. Ask for n. If the answer is a single-digit number or 'we tried it a few times', the finding is real but the number attached to it is not.

---

**Next → [C2.3 · Weight-level techniques](https://spbreed.github.io/cyber-commons/lessons/C2.3.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/C2.2.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/C2.2.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*